### Import packages, define paths and load data

In [ ]:
from pathlib import Path
import json
import spacy
import re
import gender_guesser.detector as gender
d = gender.Detector()

nlp = spacy.load("en_core_web_sm")

In [22]:
# load the system's built-in dictionary
with open('/usr/share/dict/words', 'r') as f:
    english_words = set(w.strip().lower() for w in f)

In [23]:
# Define project and data paths
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
DATA_DIR = PROJECT_ROOT / "data/cleaned"
OUTPUT_DIR = PROJECT_ROOT / "data/gendered_names"
EMBEDDINGS_DIR = PROJECT_ROOT / "embeddings"

sci_fi_data_path = DATA_DIR / "sci_fi_stories_cleaned.jsonl"
romance_data_path = DATA_DIR / "romance_stories_cleaned.jsonl"
literary_fiction_data_path = DATA_DIR / "lit_fiction_stories_cleaned.jsonl"

In [24]:
def load_jsonl(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

In [25]:
lit_fic_stories = load_jsonl(literary_fiction_data_path)
romance_stories = load_jsonl(romance_data_path)
sci_fi_stories = load_jsonl(sci_fi_data_path)

### Helper functions

In [27]:
# check if name is a common English word (to avoid misclassifying common words as names)
def is_common_word(name):
    name_lower = name.lower()
    # check singular and plural forms
    return (name_lower in english_words or 
            name_lower.rstrip('s') in english_words)  # remove plural s

# function to validate gender based on the name and the gender inferred from the pronouns in the story
def validate_gender(name, pronoun_gender):
    if name in gender_cache:
        result = gender_cache[name]
    else:
        result = d.get_gender(name)
        gender_cache[name] = result

    if is_common_word(name) and result == "unknown":
        return False
    if result in ("unknown", "andy"):
        return True
    if pronoun_gender == "male" and result in ("male", "mostly_male"):
        return True
    if pronoun_gender == "female" and result in ("female", "mostly_female"):
        return True
    return False

### Extract and classify gender

In [ ]:
# choose the most common gender identifiers
male_pronouns = {"he", "him", "his"}
female_pronouns = {"she", "her", "hers"}
gender_cache = {}

# function to extract only the given name
def clean_name(name):
    match = re.search(r'\b[A-ZÀ-ÖØ-Þ][a-zà-öø-ÿ]+\b', name)
    return match.group(0) if match else None

# function to extract names associated with male and female pronouns, and identify any names that are associated with both (conflicted)
def extract_gendered_names(stories):
    male_associated = set()
    female_associated = set()
    conflicted = set()

    for story in stories:
        text = story.get('text', str(story)).replace('\\n', '\n')
        doc = nlp(text)

        for sent in doc.sents:
            sent_lower = set(w.lower() for w in sent.text.split())
            has_male = bool(sent_lower & male_pronouns)
            has_female = bool(sent_lower & female_pronouns)

            if not (has_male ^ has_female):
                continue

            names = set()
            for ent in sent.ents:
                if ent.label_ == "PERSON":
                    clean = clean_name(ent.text.split()[0])
                    if clean:
                        names.add(clean)

            for name in names:
                if has_male and validate_gender(name, "male"):
                    if name in female_associated:
                        conflicted.add(name)
                        female_associated.discard(name)
                    elif name not in conflicted:
                        male_associated.add(name)
                elif has_female and validate_gender(name, "female"):
                    if name in male_associated:
                        conflicted.add(name)
                        male_associated.discard(name)
                    elif name not in conflicted:
                        female_associated.add(name)

    return sorted(list(male_associated)), sorted(list(female_associated)), sorted(list(conflicted))


In [29]:
male_associated_sci_fi, female_associated_sci_fi, conflicted_sci_fi = extract_gendered_names(sci_fi_stories)

In [30]:
male_associated_lit, female_associated_lit, conflicted_lit = extract_gendered_names(lit_fic_stories)

In [31]:
male_associated_romance, female_associated_romance, conflicted_romance = extract_gendered_names(romance_stories)

### Cleaning the gender lists manually

In [69]:
# manually clean the lists
# removing surnames, cities and non-names
words_to_remove = ["Activating", "Afterburn", "Aftervault", "Ascension", "Attuner", "Albuquerque", "Aristotle", "Aftershave", "Ashford", "Ackerman", "Alvarado", "Archivum", "Arkpark", "Aelions", "Alvaros", "Ambrose", "Alden", "Adler", "Antonov", "Abdi", "Abram", "Adem", "Age", "Almeida", "Alpert", "Andal", "Anord", "Ansel", "Ari", "Art", "Ate", "Auden", "Aurel", "Avenida", "Aesop", "Algés", "Alpha", "Anaïs", "Anerin", "Araujo", "Asterix", "Adebayo", "Akrasia", "Altair", "Anast", "Antigua", "Antonioa", "Aoelic", "Arana", "Arax", "Archeon", "Arcis", "Arcwyne", "Arteria", "Asteria", "Astrae", "Aurelia", "Aurion", "Azafrán",
                   "Borrowed", "Backyard", "Barbernet", "Biolab", "Bay", "Biblioteca", "Brightwater", "Buenos", "Bentley", "Bellemore", "Bellini", "Betadine", "Box", "Barnes", "Boardroom", "Brighton", "Buongiorno", "Burnham", "Bosphorus", "Bramwell", "Brylcreem", "Baudelaire", "Bellford", "Bloomfield", "Bront", "Brubeck", "Brindle", "Braeval", "Bowen", "Bach", "Babu", "Be", "Banerjee", "Batu", "Biko", "Boyd", "Budi", "Belik", "Bell", "Bhatia", "Binti", "Blakely", "Berth", "Binari", "Brontë",
                   "Chainlock", "Chordlocks", "Chronotech", "Complicit", "Cellmate", "Cappadocia", "Calderon", "Caplan", "Cavanaugh", "Cipressi", "Costello", "Calderman", "Callaghan", "Chang", "Carmichael", "Chaucer", "Carvalho", "Castillo", "Cruz", "Corbett", "Casa", "Cardoza", "Calhoun", "Calderón", "Café", "Cardenas", "Coates", "Cole", "Colfax", "Conroy", "Conway", "Crespi", "Cass", "Chekhov", "Cherry", "Cassimir", "Cerebrix", "Correia", "Cronus", "Calliope", "Cassini", "Chorale", "Constellis", "Corto", "Core",
                   "Deploying", "Driftline", "Dalloway", "Delaney", "Dickinson", "Dalrymple", "Delacroix", "Devereaux", "Diaz", "Delgado", "Devereux", "Dewitt", "Dobrev", "Duvall", "Derryhood", "Dostoevsky", "Dostoyevsky", "Dvor", "Debussy", "Doña", "Dakar", "Dalton", "Dawn", "Danvers", "Darzi", "Donovan", "Dublin", "Dahl", "Donnelly", "Dorsi", "Dunn", "Dvorák", "Dante", "Díaz", "Dalca", "Dalen", "Dhal", "Draeger",
                   "Eberhart", "Earthrise", "Echocert", "Elbridge", "Elmwood", "Ellman", "Echelion", "Ennik", "Ellery", "Elgin", "Earl", "Esposito", "Everett", "Elligson", "Eider", "Eidon", "Eremid", "Eunoia", "Eyr",
                   "Floor", "Foundries", "Frostline", "Foryou", "Freud", "Fitzgerald", "Faulkner", "Feet", "Fareeha", "Fordham", "Friedman", "Fernandes", "Ferris", "Farid", "Filmore", "Foley", "Fortuna", "Ferrer", "Façade", "Fong",
                   "Gatewatch", "Goodnight", "Grayhaven", "Glasswings", "Gardner", "Grantham", "Graybridge", "Greenway", "Gelsomini", "Graywatch", "Grierson", "Gravelle", "Gellman", "Gerber", "Glasgow", "Goldman", "Gammapier", "Glassfawn", "Gavrien", "Grosvenor", "Gustalink", "Grafton", "Gomez", "Gellar", "Gipsy", "Garza", "Ghosh", "Gran", "Gray", "Grayson", "Garrity", "Gill", "Greer", "Gutierrez", "Grouttek", "Gaianet", "Gale", "Gleason", "Governa", "Gulliver",
                   "Harmonized", "Hearthlight", "Hearthline", "Hollowpoint", "Harborline", "Harwick", "Hanford", "Harborwick", "Harrington", "Harper", "Hawke", "Hawthorne", "Headteacher", "Henderson", "Holdfasttime", "Hollis", "Halverson", "Hartwell", "Hellinger", "Harding", "Holbrook", "Halberg", "Halverton", "Hartfold", "Hensley", "Holloway", "Higgins", "Hargreeve", "Harrigan", "Harwood", "Hatfield", "Helvetica", "Hexham", "Holcomb", "Halstead", "Halvek", "Haversham", "Hoffmann", "Hollowell", "Herrera", "Hartford", "Hadley", "Hale", "Halley", "Han", "Hanley", "Hartley", "Haven", "Hostal", "Havelin", "Horvath", "Hudson", "Hwang", "Haddon", "Hald", "Haldane", "Hael", "Harto", "Hegmann", "Hiraeth", "Huldra", "Halik", "Hab", "Hanar", "Hane", "Harmony", "Haumea", "Henley",
                   "Instagram", "Ivankov", "Isla", "Ivers", "Ilyth", "Iqbal", "Izanami",
                   "Jackpoint", "Jazzmaster", "Jackpot", "Jansen", "Jefferson", "Jung", "Jovanovic", "Jamie", "Jennings", "Jesus", "Jolson", "June", "Junya", "Junebug",
                   "Keepline", "Kintsugi", "Koenig", "Kincaid", "Kjellfjord", "Kerrigan", "Kessler", "Kavinsky", "Kershaw", "Kominsky", "Klein", "Kaczynski", "Koivunen", "Kowalski", "Korvik", "Keats", "Kazi", "Kiko", "Kaltre", "Kant", "Keyes", "Khalil", "Kibo", "Konstantin", "Kramer", "Kwame", "Kraków", "Kahn", "Kallisto", "Kapoor", "Koval", "Krell", "Kura", "Kaiku", "Kairo", "Kasai", "Keelin", "Kendrew", "Keros", "Kerts", "Kessan", "Khandu", "Korr", "Kronos", "Kwan",
                   "Loopback", "Lightcall", "Langley", "Langdon", "Leclerc", "Leland", "Lopez", "Lagrange", "Langston", "Linton", "Llewellyn", "Lockhart", "Lenhart", "Lerner", "Levitt", "Lenton", "Lindstrom", "Larkfield", "Lebrun", "Lolo", "Lego", "Leica", "Levine", "Lipari", "Lonavala", "Laird", "Landry", "Lasker", "Loman", "Lowe", "Lozanos", "Lind", "Linde", "Livanos", "Lozano", "Lumenix", "López", "Luminet", "Laconia", "Larkin", "Laskin", "Lenox", "Lente", "Leong", "Lexis", "Liang", "Liko", "Ling", "Luman", "Luth", "Lysithea", "Lykos",
                   "Metadata", "Mmm", "Mnemoscientist", "Moonpool", "Mc", "Midservice", "Montefiore", "Monteverde", "Maplewood", "Marshall", "Mendez", "Minced", "Monroe", "Marchand", "Mandelstam", "Markham", "Morrisons", "Myrtle", "Menelaus", "Moreno", "Moreau", "Mao", "Marito", "Marry", "May", "Merry", "Monet", "My", "Märchen", "Maillard", "Mama", "Maram", "Marchetti", "Meyer", "Mornin", "Moses", "Málaga", "Márquez", "Marins", "Marín", "Mat", "Moon", "Marram", "Mnemo", "Maat", "Malen", "Mallor", "Marquez", "Matre", "Med", "Mek", "Miracle", "Mnem", "Mnemos", "Mnemosyn", "Morae", "Moretti", "Morrel",
                   "Newsfeeds", "Nudging", "Nocturnarium", "Negroni", "Nanogel", "Naples", "Nabokov", "Nevins", "Netwatch", "Nala", "Noor", "Nah", "Nat", "Neruda", "Nima", "Nostalgene", "Noël", "Navarro", "Nebulaa", "Nav", "Neloxa", "Neuro", "Neve", "Nine", "Nivis", "Noori", "Nostos", "Norte",
                   "Omniframe", "Onboard", "Onfoot", "Ovid", "Odysseus", "Ormsby", "Osei", "Oeuvre", "Okoye", "Olani", "Opal", "Opex", "Orlov", "Orpheid", "Orthodyne",
                   "Peacekeepers", "Plexiglas", "Patience", "Paralegal", "Palmer", "Pascal", "Perez", "Pernambuco", "Price", "Plato", "Pritchard", "Pritchett", "Poulenc", "Piper", "Papi", "Paladino", "Pierce", "Petrova", "Papá", "Paulsen", "Peralta", "Petrenko", "Peña", "Pier", "Palma", "Paredes", "Phelps", "Panthera", "Pastor", "Pelagos", "Perseus", "Palim", "Palin", "Parque", "Phrix", "Pip",
                   "Rainwater", "Remembered", "Rootroom", "Riverline", "Rachmaninoff", "Riverton", "Rodriguez", "Randall", "Rorschach", "Rotterdam", "Rianova", "Rinaldi", "Robinson", "Rosenthal", "Rivabella", "Renner", "Rafaél", "Reyes", "Roth", "Rowe", "Ramsey", "Rocha", "Rojas", "Rosen", "Renshaw", "Rosh", "Rémy", "Rahal", "Raith", "Ravik", "Rematrix", "Rhen", "Rom", "Ríos",
                   "Skydrift", "Sleepline", "Stackhouse", "Sundering", "Skywall", "Skillcaster", "Salinger", "Seabright", "Seafar", "Seafoam", "Skyraven", "Soundcheck", "Stearman", "Stonehaven", "Storycut", "Steinway", "Storycuts", "Surprised", "Shakespeare", "Strutline", "Syntelife", "Salvatore", "Schubert", "Sinatra", "Strauss", "Steele", "Seiler", "Sibelius", "Steinbeck", "Smirnov", "Stradivex", "Sverdsen", "Señora", "Seuss", "Santiago", "Santoro", "Said", "Sawicz", "Sayeed", "Signage", "Sinclair", "Siti", "Soto", "Sokol", "Stanton", "Sullivan", "Sutter", "Sato", "Señor", "Sarek", "Solomon", "Sun", "Surlan", "Salome", "Sigal", "Sol", "Solenne", "Sonarch", "Summer", "Svalin",
                   "Tennessee", "Torres", "Teenager", "Tomlin", "Tenebris", "Tectonix", "Thayer", "Tarek", "Tell", "Thorne", "Tully", "Tala", "The", "Thurs", "Toma", "Thalaea", "Takara", "Talos", "Thalen", "Thranox", "Tran", "Trask", "Tungar",
                   "Vossbacher", "Visconti", "Valencia", "Valentine", "Vega", "Velvet", "Viera", "Virginia", "Vance", "Vic", "Vicente", "Vidal", "Velez", "Varma", "Verma", "Verran", "Valek", "Valles", "Varen", "Varren", "Vass", "Venus", "Verlaine", "Vitruv", "Vonn", "Voron",
                   "Winterspine", "Whitcomb", "Whitby", "Wakecraft", "Whitaker", "Wintersong", "Winterfield", "Winnicott", "Wirt", "Willoughby", "Wroble", "Williamses", "Welles", "Wing", "Ward",
                   "You",
                   "Zookeeper", "Ziploc",
                   "Quinn",
                   "Álvarez"]



In [70]:
words_to_remove_set = set(words_to_remove)

lists = [
    "male_associated_romance",
    "female_associated_romance",
    "male_associated_lit",
    "female_associated_lit",
    "male_associated_sci_fi",
    "female_associated_sci_fi"
]

for name in lists:
    globals()[name] = [w for w in globals()[name] if w not in words_to_remove_set]

In [71]:
# remove from male_associated_sci_fi to female_associated_sci_fi
female_names_misclassified_sci_fi = ["Keita", "Keila", "Khera", "Kimara", "Rava", "Sasha", "Zia"]

male_names_misclassified_sci_fi = ["Bellamy", "Eiso", "Haruto", "Jahan", "Jools", "Jox", "Jorun", "Kavon", "Kori", "Lito", "Maison", "Murek", "Nate", "Rian", "Saito", "Thiago", "Vish"]

# delete female_names_misclassified_sci_fi from male_associated_sci_fi and add to female_associated_sci_fi
for name in female_names_misclassified_sci_fi:
    if name in male_associated_sci_fi:
        male_associated_sci_fi.remove(name)
        female_associated_sci_fi.append(name)

for name in male_names_misclassified_sci_fi:
    if name in female_associated_sci_fi:
        female_associated_sci_fi.remove(name)
        male_associated_sci_fi.append(name)

In [72]:
# check lengths of the lists
print("Romance - Male:", len(male_associated_romance))
print("Romance - Female:", len(female_associated_romance))
print("Romance - Conflicted:", len(conflicted_romance))
print("Literary Fiction - Male:", len(male_associated_lit))
print("Literary Fiction - Female:", len(female_associated_lit))
print("Literary Fiction - Conflicted:", len(conflicted_lit))
print("Sci-Fi - Male:", len(male_associated_sci_fi))
print("Sci-Fi - Female:", len(female_associated_sci_fi))
print("Sci-Fi - Conflicted:", len(conflicted_sci_fi))

Romance - Male: 251
Romance - Female: 228
Romance - Conflicted: 69
Literary Fiction - Male: 469
Literary Fiction - Female: 339
Literary Fiction - Conflicted: 131
Sci-Fi - Male: 494
Sci-Fi - Female: 419
Sci-Fi - Conflicted: 235


### Save gendered name lists

In [74]:
# Save lists
with open(OUTPUT_DIR / "male_names_sci_fi.json", "w") as f:
    json.dump(male_associated_sci_fi, f)

with open(OUTPUT_DIR / "female_names_sci_fi.json", "w") as f:
    json.dump(female_associated_sci_fi, f)

with open(OUTPUT_DIR / "male_names_lit.json", "w") as f:
    json.dump(male_associated_lit, f)   

with open(OUTPUT_DIR / "female_names_lit.json", "w") as f:
    json.dump(female_associated_lit, f)

with open(OUTPUT_DIR / "male_names_romance.json", "w") as f:
    json.dump(male_associated_romance, f)

with open(OUTPUT_DIR / "female_names_romance.json", "w") as f:
    json.dump(female_associated_romance, f)